In [2]:
class FileRedactor:
    "Class which can remove lines from a file based on a list of keywords/phrases."
    "Essentially takes the text from the BARS questionnaire and redacts lines from the"
    "reports which contain one of the lines, or are a portion of the line."
    "Not especially reliable, but should be good enough for testing purposes."
    def __init__(self, file, redaction_file, output, log):
        self.file = file
        self.redaction_file=redaction_file
        self.log=log
        self.lines_removed=None

        self.lines_to_redact=[]
        self.lines_to_redact_stripped=[]
        with open(redaction_file, 'r') as to_redact:
            for line in to_redact:
                self.lines_to_redact.append(line)
                self.lines_to_redact_stripped.append(''.join([char for char in line if char.isalpha()]))

        self.output=output
            
    def file_reader(self,file):
        """Generator to read the input file line by line."""
        for row in open(file, "r", encoding='utf-8'):
            yield row
    
    def strip_non_alpha(self, string):
        return ''.join([char for char in string if (char.isalpha() or char in '0123456789') ])
    
    def redact_lines(self):
        reader=self.file_reader(self.file)
        self.lines_removed=0
        for row in reader:
            if row=='\n':
                with open(self.output, 'a') as out:
                    out.write(row)
                continue
            stripped=self.strip_non_alpha(row)
            good=True
            check_list=self.lines_to_redact_stripped +['0: Normal']
            for substring in check_list:
                if stripped.startswith('gait') or stripped.startswith('Gait'):
                    # print('ok')
                    good=False
                    break
                if stripped.startswith('speech') or stripped.startswith('Speech'):
                    # print('ok')
                    good=False
                    break
                if substring in stripped:
                    if substring.lower() in ['gait', 'speech'] and len(row.strip())>8:
                        continue
                    good=False
                    # row=row[:-1]+f'  |   bars phrase"{substring}" in line\n'
                    break
                elif stripped==substring:
                    # row=row[:-1]+'  |   =bars phrase\n'
                    good=False
                    break
                elif stripped in substring and len(stripped)>7:
                    # row=row[:-1]+'  |   line is part of bars phrase '+substring+'\n'
                    good=False
                    break

                # if  substring[:20] in stripped:
                #     good=False
                #     break
                # if ( and stripped in substring) and stripped.count(' ')!=0:
                #     good=False
                #     break
                # if row[:3]!='   ' and stripped in substring:
                #     good=False
                #     break

            if good:
                with open(self.output, 'a') as out:
                    out.write(row)
            else:
                with open(self.log, 'a') as mylog:
                    mylog.write(row)
                    self.lines_removed+=1


In [3]:
input='/Users/rm026/Documents/code/ReviewPyper_testing/msa_prg_and_dis_deidentified.txt'
output='/Users/rm026/Documents/code/ReviewPyper_testing/bars_redaction/msa_prg_and_dis_deidentified_bars_redacted_test.txt'
to_redact='/Users/rm026/Documents/code/ReviewPyper_testing/bars_redaction/bars_stuff_to_remove.txt'
removal_log='/Users/rm026/Documents/code/ReviewPyper_testing/bars_redaction/redaction_log.txt'


In [4]:
redactor=FileRedactor(input, to_redact, output, removal_log)
redactor.redact_lines()

In [5]:
# import os 
# root_folder='/Users/rm026/Documents/code/ReviewPyper_testing/bars_redaction/individual_notes/'
# inputs=[file for file in os.listdir(root_folder) if not file.startswith('.')]

In [6]:
# removed={}
# for file in inputs:
#     output=root_folder+file.replace('.txt','_redacted.txt')
#     redaction_log=root_folder+file.replace('.txt','_redaction_log.txt')
#     redactor=FileRedactor(root_folder+file, to_redact, output, redaction_log)
#     redactor.redact_lines()
#     removed[file]=redactor.lines_removed
# removed